# 📦 Libraries and Tools
This cell loads the tools we need to read files, handle image data, save memory files, and run the PCA model.

In [ ]:
# 📂 Load system tools for reading files
import os
import pickle

# 🔢 Load tool for working with numbers and arrays
import numpy as np

# 🖼️ Load tool for opening and resizing images
from PIL import Image

# 🤖 Load PCA tool to find main face patterns (Eigenfaces)
from sklearn.decomposition import PCA

print("✅ Tools loaded successfully!")

✅ Tools loaded successfully!


# ⚙️ Folder Names and Settings
This cell sets the names of our image folders, the memory file, and the matching limit.

In [ ]:
# 📁 Set folder paths for training and testing
train_folder = "images"          # 🎯 Folder with known training faces
test_folder = "test"            # 🧪 Folder with test faces (Target person)
test_unknown_folder = "test_unknown"  # ❓ Folder with test faces (Other people 1)
test_unknown2_folder = "test_unknown2" # ❓ Folder with test faces (Other people 2)

# 💾 File name to save trained memory
model_file = "eigenface_memory.pkl"

# 📏 Match limit: lower distance means faces look more similar
threshold = 1.9

print("⚙️ Setup completed!")

# 🧠 Learn Faces and Create PCA Subspace
This cell opens training images, resizes them to 64x64, turns them into numbers, and trains the PCA model to extract Eigenfaces.

In [ ]:
# 📸 List to hold image pixel data
faces_list = []

# 📥 Read images from training folder
if os.path.exists(train_folder):
    for filename in os.listdir(train_folder):
        img_path = os.path.join(train_folder, filename)
        try:
            with Image.open(img_path) as img:
                # 🎨 Change image to black and white, and resize to 64x64
                gray_img = img.convert('L').resize((64, 64))
                # 📐 Turn 2D image into 1D list of numbers
                faces_list.append(np.array(gray_img).flatten())
        except Exception as e:
            print(f"⚠️ Could not read file {filename}: {e}")

# 📊 Combine all images into one big list of numbers
X_train = np.array(faces_list)

# 🛑 Check if we have images to train
if len(X_train) == 0:
    print("❌ Error: No images found in 'images/' folder!")
    exit()

# 🧮 Choose number of main patterns to learn
n_components = min(X_train.shape[0], 10)

# 🔬 Train PCA on images
pca = PCA(n_components=n_components, whiten=True).fit(X_train)

# 📐 Convert training images into small weight numbers
X_train_pca = pca.transform(X_train)

print(f"🚀 Training done! Learned {n_components} main face patterns.")

# 🖼️ Print All Eigenfaces in One Image
This separate cell displays all learned Eigenfaces together in a single grid image.

In [ ]:
# 🖼️ Display all Eigenfaces together in one figure grid
print("🖼️ Printing all Eigenfaces in one image...")
fig, axes = plt.subplots(1, n_components, figsize=(n_components * 2, 3))

# Handle single image axis case
if n_components == 1:
    axes = [axes]

for i in range(n_components):
    # Reshape 1D vector back to 64x64 2D image
    eigenface_img = pca.components_[i].reshape(64, 64)
    axes[i].imshow(eigenface_img, cmap='gray')
    axes[i].set_title(f"Eigenface #{i+1}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

# 💾 Save Memory to File
This cell saves our trained PCA model and training weight numbers into a file on disk.

In [ ]:
# 📦 Put model and weight data together
memory_data = {
    'pca': pca,
    'X_train_pca': X_train_pca
}

# 💾 Save data into a binary file
with open(model_file, 'wb') as f:
    pickle.dump(memory_data, f)

print(f"💾 Memory saved to '{model_file}' successfully!")

# 📂 Read Saved Memory
This cell reads the saved memory file from disk back into Python.

In [ ]:
# 🔓 Open and read the memory file
with open(model_file, 'rb') as f:
    loaded_memory = pickle.load(f)

# 🔑 Extract the saved model and weight data
loaded_pca = loaded_memory['pca']
loaded_train_pca = loaded_memory['X_train_pca']

print("🧠 Memory loaded from disk successfully!")

# 🧪 Prepare for Testing
This cell sets up and prepares the test function to check images against our saved memory.

In [ ]:
def run_test_pass(folder_path, pass_name):
    """
    🧪 Function to check if images in a folder match saved memory.
    """
    test_faces = []

    # 📥 Load images from the given folder
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)
            try:
                with Image.open(img_path) as img:
                    gray_img = img.convert('L').resize((64, 64))
                    test_faces.append(np.array(gray_img).flatten())
            except Exception:
                pass

    X_test = np.array(test_faces)

    # ⚠️ Check if folder is empty
    if len(X_test) == 0:
        print(f"⚠️ Warning: No images found in '{folder_path}'.")
        return False

    # 📐 Convert test images into weight numbers using saved model
    X_test_pca = loaded_pca.transform(X_test)
    all_passed = True

    print(f"=== 🔍 Checking: {pass_name} (Folder: '{folder_path}') ===")

    # 📏 Check distance for each test image
    for test_vector in X_test_pca:
        # 📐 Calculate distance to saved training memory
        distances = np.linalg.norm(loaded_train_pca - test_vector, axis=1)
        min_dist = np.min(distances)

        # 🎯 If distance is too big, mark as failed
        if min_dist >= threshold:
            all_passed = False

    # 📊 Print summary result
    print("-------------------------")
    if all_passed:
        print(f"🏆 Result for {pass_name}: PASSED")
    else:
        print(f"❌ Result for {pass_name}: FAILED")
    print("-------------------------\n")

    return all_passed

# 🟢 Test 1: Known Target Person
This cell tests images from the main test folder to verify if they match the trained memory (expected to PASS).

In [ ]:
# 🟢 Run test on the known target identity folder
run_test_pass(test_folder, "Test 1 (Known Person)")

# 🔴 Test 2: Unknown Identity Group 1
This cell tests images from the test_unknown folder to verify if non-target faces are correctly rejected (expected to FAIL if they do not match).

Cell 18: Code

In [ ]:
# 🔴 Run test on the first unknown identity folder
run_test_pass(test_unknown_folder, "Test 2 (Unknown Group 1)")

# 🔴 Test 3: Unknown Identity Group 2
This cell tests images from the test_unknown2 folder to verify rejection accuracy on another set of non-target faces (expected to FAIL if they do not match).

In [ ]:
# 🔴 Run test on the second unknown identity folder
run_test_pass(test_unknown2_folder, "Test 3 (Unknown Group 2)")